# VoiceHaul - long-horizon evaluation for empathic voice agents

**Runs entirely in this notebook. No API key, no account, nothing to install.**
Press *Runtime -> Run all* and read down the page; it takes about a minute.

A turn-level rating tells you whether a response *sounded* right. It cannot tell you
whether a model still honours what the user asked for twenty turns ago, whether its
calibration decays as a session runs long, whether it is regulating the user's affect
or just mirroring it back, or which turn broke a call that ended badly.

This notebook measures those four things, and then measures how much rating budget you
need before any of them is detectable.

---


## 1. Get the code


In [ ]:
!git clone -q https://github.com/vahit19/voicehaul.git 2>/dev/null || echo 'already cloned'
%cd -q voicehaul
!ls


## 2. Check the harness before trusting its output

An evaluation harness that is wrong is worse than no harness, because it is believed.
These fifteen property checks run against the one case where the answer is known by
construction: the oracle policy really is the ceiling, obeying a user request really
costs no calibration, the injected fault really is recovered.


In [ ]:
!python test_voicehaul.py


## 3. The full report

Every number below is computed from this run. Nothing is hard-coded.


In [ ]:
!python run_demo.py


## 4. The four charts


In [ ]:
from IPython.display import Image, display
display(Image('voicehaul_report.png'))


## 5. Poke at it yourself

The result that matters most is the first one: a fixed-context turn panel and a real
conversation rank the same five models in nearly opposite orders. Below you can watch
one conversation happen turn by turn and see why.

Change AGENT to MirrorAgent, FlatAgent, DrifterAgent or CalibratedAgent and re-run.
Watch the distress column.


In [ ]:
from voicehaul.agents import MirrorAgent, FlatAgent, DrifterAgent, CalibratedAgent
from voicehaul.env import PERSONAS
from voicehaul.runner import run_episode

AGENT   = MirrorAgent      # <- change me
PERSONA = PERSONAS[1]      # 0 billing, 1 hostile, 2 confused, 3 grieving, 4 calm

ep = run_episode(AGENT(), PERSONA, seed=3, n_turns=40)

print('persona: {}   agent: {}'.format(ep.persona, ep.agent))
print()
print('turn  distress  calib  perceived  rate  cheer   ack   user said')
print('-' * 78)
for t in ep.turns:
    said = ''
    if t.new_directive:
        said = t.new_directive.replace('_', ' ')
    if t.shock > 0:
        said = ('(new grievance) ' + said).strip()
    print('{:>4}  {:>8.2f}  {:>5.2f}  {:>9.2f}  {:>4.2f}  {:>5.2f}  {:>5.2f}  {}'.format(
        t.index, t.user_after.negative_load, t.calibration, t.perceived,
        t.action.speech_rate, t.action.cheerfulness, t.action.acknowledgement, said))
print()
print('conversation failed:', ep.failed)


### What to look for

With MirrorAgent on the hostile persona, calibration and perceived empathy both stay
respectable the whole way down - it sounds attuned on every single turn - while
distress never comes down. It matches the user's energy instead of sitting below it,
so the user has nowhere to come down to. No turn is bad. The conversation is.

With DrifterAgent, watch what happens after a slow-down request: it complies for a
while, then quietly stops, and the speech rate creeps back up. The turn-level score
barely moves.


## 6. Find the turn a regression started

A fault is injected at a turn you can see below; the diagnostic never gets told where.
Cheap deterministic signals propose candidates, a segment walk-back finds where the
anomalous stretch begins, and counterfactual replay refuses to answer if repairing the
agent from that turn would not have changed the outcome.

Lower SEVERITY to blend the fault into the model's own policy - that is the realistic
and much harder case.


In [ ]:
from voicehaul.onset import localize, anomaly_scores

SEVERITY   = 1.0    # 1.0 = full reversion, 0.35 = subtle and realistic
FAULT_TURN = 18

ep = run_episode(CalibratedAgent(), PERSONAS[0], seed=42, n_turns=40,
                 fault_turn=FAULT_TURN, fault_severity=SEVERITY)
onset, ranked = localize(ep, PERSONAS[0])

print('true fault turn  :', ep.true_fault_turn)
print('predicted onset  :', onset)
print('ranked candidates:', ranked[:5])
print()
scores = anomaly_scores(ep)
for i in range(max(0, FAULT_TURN - 6), min(len(scores), FAULT_TURN + 7)):
    bar = '#' * int(scores[i] * 18)
    mark = '   <- fault injected here' if i == FAULT_TURN else ''
    print('turn {:>2}  {:>5.2f}  {}{}'.format(i, scores[i], bar, mark))


## 7. How much rating budget does a regression suite need?

Human ratings are both the ground truth and the budget line. This is the calculation
that turns 'we track regressions' into a number of conversations and a cost.


In [ ]:
from voicehaul.metrics import min_detectable_effect, required_n

SIGMA_BETWEEN = 0.40   # between-conversation spread, measured from the suite
RATER_SIGMA   = 0.90   # per-rater noise, in Likert points

print('smallest regression detectable at 80% power, in Likert points')
print()
print('{:<22}{:>9}{:>9}{:>9}{:>9}'.format(
    'raters/conversation', 'N=30', 'N=100', 'N=300', 'N=1000'))
for nr in (1, 3, 5, 10):
    row = ['{:.2f}'.format(min_detectable_effect(SIGMA_BETWEEN, RATER_SIGMA, n, nr))
           for n in (30, 100, 300, 1000)]
    print('{:<22}{:>9}{:>9}{:>9}{:>9}'.format(nr, *row))
print()
for target in (0.50, 0.30, 0.20, 0.10):
    print('to catch a {:.2f}-point regression with 3 raters: N = {:>5} conversations'
          .format(target, required_n(target, SIGMA_BETWEEN, RATER_SIGMA, 3)))


---

## What is real and what is simulated

**Simulated:** the users, the agents, and the affect dynamics. The five agents are
deterministic policy simulators, not language models - each embodies exactly one known
failure mode. The user model and both scoring functions are written by hand.

**Real:** the metrics, the estimators, the statistics, and the localization algorithm.
Those are the deliverable.

The reason for a synthetic environment is not convenience. **You cannot validate a
measurement instrument without ground truth you control.** If you only ever run an eval
against real models, a metric that reports the wrong thing and a model that behaves
badly are indistinguishable. Here the fault turn is known, the failure mode is known,
and the ideal policy is known - so '93% accurate at severity 0.5 with no false
positives' is a checkable statement about the *method*, not a leaderboard entry.

voicehaul/adapters.py holds the seams where a real speech-to-speech model, a real
expression-measurement source and a real human rating panel plug in. The design
property that makes it practical: the harness needs no privileged access to the model
under test - both sides of the conversation are scored from audio.

---

Vahit Feryad - [Runopsy](https://github.com/vahit19/runopsy) -
[LongHaul-Bench](https://github.com/vahit19/LongHaul-Bench)
